### Curado final de los espectros

Este código realiza un último filtrado. En caso de que el espectro tenga una linea continua en el medio (no saltos vacíos), los descarta. Para esto, se analiza **cuatro** valores consecutivos en lambda (se puede modificar este valor)



In [ ]:
import os
import numpy as np
from astropy.io import fits

In [ ]:
from google.colab import drive
drive.mount('/drive')

Mounted at /drive


In [ ]:
#path a las carpetas
cleaned_dir = '/drive/My Drive/Data/spec_data/cleaned_spec/' #carpeta donde estan los espectros cortados y que pasaron el primer filtro

output_dir = '/drive/My Drive/Data/spec_data/final_spec/'  #carpeta donde se guardarán los espectros que no se descartan (los espectros finales)
os.makedirs(output_dir, exist_ok=True)

In [ ]:
col_loglam = 'loglam'
col_flux = 'flux'
flux_dif_threshold = 0.002  #diferencia entre los valores de flux (límite permitido)

In [ ]:
#crear un lista con los espectros a procesar (en este caso en cleaned_dir)
fits_files = [os.path.join(cleaned_dir, file) for file in os.listdir(cleaned_dir)
    if file.endswith('.fits')]

In [ ]:
#contador para saber la cantidad de espectros descartados
discarded_count = 0

#procesamiento de los espectros que no pasen el filtro

for fits_file in fits_files:
    #abrir los espectros (.fits)
    hdul = fits.open(fits_file)
    data = hdul[1].data

    #extraer valores de loglam y flux, y guardarlos
    loglam_values = data[col_loglam]
    flux_values = data[col_flux]

    #analizar los cuatro valores consecutivos de loglam
    discarded = False
    invalid_loglam_flux_pairs = []
    for i in range(len(loglam_values) - 3):  #si quiero analizar mas valores, cambio estos numeros (3, 4, 4)
        loglam_block = loglam_values[i:i+4]
        flux_block = flux_values[i:i+4]

        #verificar diferencias entre los valores de flux
        diffs = [abs(flux_block[j+1] - flux_block[j]) for j in range(3)]  #un for para hacer diferencias entre los valores

        #analiza la condición: si alguna de las diferencias es menor o igual al límite, se descarta
        if all(diff <= flux_dif_threshold for diff in diffs):
            discarded = True
            invalid_loglam_flux_pairs.append((loglam_block, flux_block))

    if discarded: #este bloque indica cual espectro fue descartado y cuales loglam con sus respectivos valores de flujo que no cumplen con la condición
        discarded_count += 1
        print(f'\nEl espectro {fits_file} fue descartado')

    else:
        #guardar el espectro si pasa el filtro

        #cambiar el nombre del archivo
        #quitarle el 'cortado_'
        new_name = os.path.basename(fits_file).replace('cleaned_', '')

        #agregar el prefijo "cleaned_"
        output_file = os.path.join(output_dir, f'final_{new_name}')
        #eliminar el archivo si ya existe, para forzar sobreescritura
        if os.path.exists(output_file):
            os.remove(output_file)

        #guardar el archivo
        hdul.writeto(output_file)
        #print(f'El espectro {fits_file} pasa el filtro y se guardó en {output_file}.')

    hdul.close()

#imprimir la cantidad de espectros descartados
print(f'\nNúmero total de espectros descartados: {discarded_count}')


Número total de espectros descartados: 0


In [ ]:
final_files = len(os.listdir(output_dir))
print(f'Hay {final_files} archivos en la carpeta {output_dir}')

Hay 121 archivos en la carpeta /drive/My Drive/Data/spec_data/final_spec/
